# Model images and Blender notebook rendering
Inspect **22 preserved synthetic comparison cases**, display their saved renders,
and optionally render the geometry from a notebook cell in Blender.
The default uses cached images and runs in ordinary Jupyter without Blender.

For live editing, use [Jan-Hendrik Müller's Jupyter extension](https://github.com/kolibril13/jupyter-blender)
and the [setup guide](../docs/blender-notebooks.md). It provides a kernel inside Blender
so `import bpy` works. The integration uses the public Blender API and does not vendor
or automatically install the extension. DiffeRT and Sionna stay in their own environments.


In [ ]:
from pathlib import Path
import json
import sys
# Set to the checkout directory if the Jupyter server starts elsewhere.
ROOT_OVERRIDE = None
ROOT = Path(ROOT_OVERRIDE).resolve() if ROOT_OVERRIDE else next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks/visuals.py").is_file())
sys.path.insert(0, str(ROOT / "notebooks"))
from IPython.display import display, Markdown, Image
from visuals import verify_assets, compare_renders, ASSETS
manifest = verify_assets()
models = json.loads((ASSETS / "models.json").read_text())["models"]
display(Markdown("| Case | Frequency | Meshes |\n| --- | ---: | ---: |\n" + "\n".join(
    f"| `{key}` | {m['frequency_hz']/1e9:g} GHz | {len(m['shapes'])} |" for key, m in models.items())))


## Match the camera before comparing paths
These views share geometry, framing, lighting and resolution. Only the selected
retained path differs. Green surfaces are leaf screens; brown solids are square
wood columns. Red = TX, blue = RX, orange = retained path. Marker thickness is
enlarged for readability. Blender colors are illustrative, not radio material values.


In [ ]:
left = ASSETS / "blocked_sionna.png"
right = ASSETS / "blocked_differt.png"
a, b = [json.loads(p.with_suffix(".json").read_text()) for p in (left, right)]
assert a["case"] == b["case"] and a["camera"] == b["camera"]
assert a["geometry_sha256"] == b["geometry_sha256"]
display(compare_renders(left, right,
    ("Sionna RT: strongest retained V path", "DiffeRT: strongest retained V path")))


## Optional live render with Jan-Hendrik's extension
Open this notebook from the extension's Jupyter server in Blender 5.1+.
Set `RENDER_LIVE = True` below, then run the cells. A separate scene is created for
each render; the current scene and its render settings are preserved. In Blender's
scene selector choose the new **DiffeRT Notebook** scene to inspect it.

Choose any listed `CASE`. `PATH_INDEX` applies only to `multiple/blocked_coupled`:
0 = strongest Sionna, 1 = strongest DiffeRT, 2 = D-R-D, 3 = four diffractions.
`DETAIL = True` fits geometry only, so distant TX/RX can be outside the frame.
A presentation render does not run physics or update the historical comparison.


In [ ]:
RENDER_LIVE = False
CASE = "multiple/blocked_coupled"
PATH_INDEX = 0  # set None for cases without a saved path
DETAIL = False
OUTPUT = ROOT / "build/notebook-renders"
if RENDER_LIVE:
    import bpy
    from blender_render import render_model
    receipt = render_model(CASE, OUTPUT / "selected.png", path_index=PATH_INDEX, detail=DETAIL)
    display(Image(filename=str(OUTPUT / "selected.png")))
    print(receipt)
else:
    print("Cached images displayed. Enable RENDER_LIVE inside Blender for a fresh presentation render.")


## Combine two fresh renders inline
This cell uses the identical model and camera for the two paths, and embeds the
PNGs into the notebook output. The exported HTML remains readable without Blender,
an external image server or a widget extension. The bundled data and historical
numerical results remain unchanged.


In [ ]:
if RENDER_LIVE:
    receipts = [render_model("multiple/blocked_coupled", OUTPUT / f"paired_{i}.png", path_index=i)
                for i in (0, 1)]
    assert receipts[0]["camera"] == receipts[1]["camera"]
    display(compare_renders(OUTPUT / "paired_0.png", OUTPUT / "paired_1.png"))
else:
    display(compare_renders(left, right))


## Provenance and scope
[Asset manifest](assets/manifest.json) checks every figure, geometry extract and render
receipt. [Asset notes](assets/README.md) explain the source studies and display choices.
Geometry coordinates and triangle indices are preserved; no private actual-plant
diagnostic scene is included. The original 13 numerical input files are unchanged.
The 2D full-wave maps in notebook 03 are preserved solver figures, not Blender renders.
